##  comparison between a fully connected deep neural network (DNN) & DeepMetabio-mCRC Screener

In [2]:
# =========================================================
# CNN vs DNN Cross-Validation Training Script
# Purpose:
# - Train 1D-CNN (proposed model)
# - Train Fully Connected DNN (MLP baseline)
# - Use identical data, features, splits, CV strategy
# =========================================================

import os
import random
import json
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Dropout, Input,
    Conv1D, MaxPooling1D, Flatten
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

e:\conda-envs\tensorflow\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
# =========================================================
# Step 0: Global settings
# =========================================================

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE_DIR = "D:\结直肠癌肝转移Biomarker 诊断\投稿\Computational and Structural Biotechnology\Ready for Submit\For Submit\返修\返修数据与脚本"
EXPRESSION_FILE = os.path.join(BASE_DIR, "expression_data_combat_corrected.csv")
METADATA_FILE = os.path.join(BASE_DIR, "metadata_combined.csv")
GENE_FILE = os.path.join(BASE_DIR, "functional_genes_620.txt")

OUTPUT_DIR = os.path.join(BASE_DIR, "cnn_vs_dnn_cv_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Environment initialized")

✅ Environment initialized


In [4]:
# =========================================================
# Step 1: Load data
# =========================================================

expr = pd.read_csv(EXPRESSION_FILE, index_col=0)
meta = pd.read_csv(METADATA_FILE, index_col=0)
meta = meta.loc[expr.index]

with open(GENE_FILE) as f:
    genes = [g.strip() for g in f if g.strip()]

genes = [g for g in genes if g in expr.columns]

X = expr[genes]
y_raw = meta["group"]

le = LabelEncoder()
y = le.fit_transform(y_raw)

print(f"Samples: {X.shape[0]}, Genes: {X.shape[1]}")
print("Label mapping:", dict(zip(le.classes_, range(len(le.classes_)))))

Samples: 1077, Genes: 606
Label mapping: {'metastasis': 0, 'primary': 1}


In [5]:
# =========================================================
# Step 2: Train / internal split (same for CNN & DNN)
# =========================================================

X_pool, X_final, y_pool, y_final = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=SEED
)

print(f"CV pool: {X_pool.shape[0]}")
print(f"Internal test set: {X_final.shape[0]}")

CV pool: 861
Internal test set: 216


In [6]:
# =========================================================
# Step 3: Define models
# =========================================================

def create_cnn_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        Conv1D(32, 5, activation="relu"),
        MaxPooling1D(2),
        Dropout(0.3),
        Conv1D(64, 5, activation="relu"),
        MaxPooling1D(2),
        Dropout(0.4),
        Flatten(),
        Dense(64, activation="relu"),
        Dropout(0.5),
        Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model


def create_dnn_model(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(256, activation="relu"),
        Dropout(0.5),
        Dense(128, activation="relu"),
        Dropout(0.5),
        Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [7]:
# =========================================================
# Step 4: 5-fold Stratified CV (CNN & DNN in parallel)
# =========================================================

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

results = {
    "CNN": [],
    "DNN": []
}

cnn_models, cnn_scalers = [], []
dnn_models, dnn_scalers = [], []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_pool, y_pool), 1):
    print(f"\n=== Fold {fold} ===")

    X_tr = X_pool.iloc[tr_idx]
    X_val = X_pool.iloc[val_idx]
    y_tr = y_pool[tr_idx]
    y_val = y_pool[val_idx]

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_val_s = scaler.transform(X_val)

    # ---------- CNN ----------
    X_tr_cnn = np.expand_dims(X_tr_s, axis=-1)
    X_val_cnn = np.expand_dims(X_val_s, axis=-1)

    cnn = create_cnn_model((X_tr_cnn.shape[1], 1))
    cnn.fit(
        X_tr_cnn, y_tr,
        epochs=50,
        batch_size=32,
        validation_data=(X_val_cnn, y_val),
        callbacks=[
            EarlyStopping(patience=10, restore_best_weights=True),
            ReduceLROnPlateau(patience=5)
        ],
        verbose=0
    )

    y_val_proba = cnn.predict(X_val_cnn, verbose=0).flatten()
    results["CNN"].append({
        "AUC": roc_auc_score(y_val, y_val_proba),
        "Accuracy": accuracy_score(y_val, y_val_proba > 0.5),
        "Precision": precision_score(y_val, y_val_proba > 0.5),
        "Recall": recall_score(y_val, y_val_proba > 0.5)
    })

    cnn_models.append(cnn)
    cnn_scalers.append(scaler)

    # ---------- DNN ----------
    dnn = create_dnn_model(X_tr_s.shape[1])
    dnn.fit(
        X_tr_s, y_tr,
        epochs=50,
        batch_size=32,
        validation_data=(X_val_s, y_val),
        callbacks=[
            EarlyStopping(patience=10, restore_best_weights=True),
            ReduceLROnPlateau(patience=5)
        ],
        verbose=0
    )

    y_val_proba = dnn.predict(X_val_s, verbose=0).flatten()
    results["DNN"].append({
        "AUC": roc_auc_score(y_val, y_val_proba),
        "Accuracy": accuracy_score(y_val, y_val_proba > 0.5),
        "Precision": precision_score(y_val, y_val_proba > 0.5),
        "Recall": recall_score(y_val, y_val_proba > 0.5)
    })

    dnn_models.append(dnn)
    dnn_scalers.append(scaler)


=== Fold 1 ===

=== Fold 2 ===

=== Fold 3 ===

=== Fold 4 ===

=== Fold 5 ===


In [8]:
# =========================================================
# Step 5: Summarize CV results
# =========================================================

summary = {}
for model_type in ["CNN", "DNN"]:
    df = pd.DataFrame(results[model_type])
    summary[model_type] = {
        "Mean_AUC": df["AUC"].mean(),
        "Std_AUC": df["AUC"].std(),
        "Mean_Accuracy": df["Accuracy"].mean(),
        "Mean_Precision": df["Precision"].mean(),
        "Mean_Recall": df["Recall"].mean()
    }

print("\n=== Cross-validation summary ===")
print(json.dumps(summary, indent=4))

with open(os.path.join(OUTPUT_DIR, "cnn_vs_dnn_cv_summary.json"), "w") as f:
    json.dump(summary, f, indent=4)


=== Cross-validation summary ===
{
    "CNN": {
        "Mean_AUC": 0.9112938267708252,
        "Std_AUC": 0.018204567052785805,
        "Mean_Accuracy": 0.8548259174620245,
        "Mean_Precision": 0.8652999947672549,
        "Mean_Recall": 0.9268192219679634
    },
    "DNN": {
        "Mean_AUC": 0.9262602909071781,
        "Std_AUC": 0.023644440866559063,
        "Mean_Accuracy": 0.8792445221131873,
        "Mean_Precision": 0.8948985707148516,
        "Mean_Recall": 0.928604118993135
    }
}


In [9]:
# =========================================================
# Save models (optional)
# =========================================================

MODEL_DIR = os.path.join(OUTPUT_DIR, "ensemble_models")
os.makedirs(MODEL_DIR, exist_ok=True)

for i, (m, s) in enumerate(zip(cnn_models, cnn_scalers), 1):
    m.save(os.path.join(MODEL_DIR, f"cnn_fold{i}.keras"))
    joblib.dump(s, os.path.join(MODEL_DIR, f"cnn_scaler{i}.pkl"))

for i, (m, s) in enumerate(zip(dnn_models, dnn_scalers), 1):
    m.save(os.path.join(MODEL_DIR, f"dnn_fold{i}.keras"))
    joblib.dump(s, os.path.join(MODEL_DIR, f"dnn_scaler{i}.pkl"))

joblib.dump(le, os.path.join(MODEL_DIR, "label_encoder.pkl"))

print("\n🎉 CNN vs DNN training completed successfully.")


🎉 CNN vs DNN training completed successfully.
